# 03_model.ipynb — Model Training and Evaluation

**Purpose:** Single unified LOSO loop across 32 subjects, 3 targets, 3 feature configs, 2 classifiers. Followed by batched null distribution with checkpoint saving.

**Inputs:** `results/X_features.npy`, `results/X_peripheral_features.npy`, `results/y.npy`, `results/subject_ids.npy`

**Outputs:** `results/loso_results.pkl`, `results/importance_agg.pkl`, `results/importance_corr.pkl`, `results/importance_mean_*.npy`, `results/null_results.pkl`

**Settings (finalized after trade-off analysis):**
- `n_jobs=-3` — all cores minus 2, thermal safety
- `rf_n_estimators=150` — stable band-level and FAA importance, 25% faster than 200
- `n_perm_repeats=20` — stable on 40-sample test set after 32-fold averaging
- `float32` — halves memory bandwidth, no effect on results at this precision

**Run once. Do not re-run unless results are corrupted.**

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pickle
from pathlib import Path

from models import (
    run_loso, aggregate_importance,
    importance_correlation, shuffled_label_null,
    TARGETS, BAND_NAMES, CONFIGS, CORR_KEYS,
)

Path('../results').mkdir(exist_ok=True)

# Load and immediately cast to float32
X_features            = np.load('../results/X_features.npy').astype(np.float32)
X_peripheral_features = np.load('../results/X_peripheral_features.npy').astype(np.float32)
y                     = np.load('../results/y.npy')            # keep float64 for median computation
subject_ids           = np.load('../results/subject_ids.npy')

print(f"X_features:            {X_features.shape}  dtype={X_features.dtype}")
print(f"X_peripheral_features: {X_peripheral_features.shape}  dtype={X_peripheral_features.dtype}")
print(f"y:                     {y.shape}")
print(f"subject_ids:           {subject_ids.shape}")

X_features:            (1280, 131)  dtype=float32
X_peripheral_features: (1280, 8)  dtype=float32
y:                     (1280, 4)
subject_ids:           (1280,)


## 1. LOSO loop

All 32 folds run in parallel. joblib verbose=10 prints fold completion as they finish.
Expected runtime: 8-15 min depending on hardware.

In [2]:
results = run_loso(
    X_eeg            = X_features,
    X_peripheral     = X_peripheral_features,
    y                = y,
    subject_ids      = subject_ids,
    n_perm_repeats   = 20,
    rf_n_estimators  = 150,
    n_jobs           = -3,
)
print("LOSO complete.")

LOSO folds:   0%|          | 0/32 [00:00<?, ?fold/s]

[FLAG] valence: imbalance >65/35 in subjects [3, 4, 6, 12, 17, 23, 25, 26, 27, 28, 29, 31]
[FLAG] arousal: imbalance >65/35 in subjects [2, 4, 8, 9, 15, 20, 23, 24, 26, 27, 28, 32]
[FLAG] dominance: imbalance >65/35 in subjects [4, 9, 16, 18, 20, 25, 27, 30, 32]
LOSO complete.


## 2. Accuracy sanity check

Chance = 50%. Below 52% = bug. Published DEAP cross-subject baselines: 55-75%.

In [3]:
print(f"{'Target':<12} {'Config':<12} {'Model':<6} {'Acc':>6} {'Std':>6} {'F1':>6}")
print('-' * 52)
for target in TARGETS:
    for cfg in CONFIGS:
        for model in ('svm', 'rf'):
            acc = results['metrics'][target][cfg][model]['acc']
            f1  = results['metrics'][target][cfg][model]['f1']
            flag = ' <BUG' if np.mean(acc) < 0.50 else '' 
            print(f"{target:<12} {cfg:<12} {model:<6} "
                  f"{np.mean(acc):>6.3f} {np.std(acc):>6.3f} {np.mean(f1):>6.3f}{flag}")

Target       Config       Model     Acc    Std     F1
----------------------------------------------------
valence      eeg          svm     0.558  0.159  0.512
valence      eeg          rf      0.527  0.105  0.446
valence      peripheral   svm     0.497  0.126  0.504 <BUG
valence      peripheral   rf      0.499  0.090  0.475 <BUG
valence      combined     svm     0.555  0.155  0.500
valence      combined     rf      0.537  0.132  0.467
arousal      eeg          svm     0.500  0.136  0.420
arousal      eeg          rf      0.537  0.122  0.357
arousal      peripheral   svm     0.538  0.093  0.571
arousal      peripheral   rf      0.530  0.090  0.521
arousal      combined     svm     0.536  0.110  0.488
arousal      combined     rf      0.527  0.104  0.369
dominance    eeg          svm     0.471  0.156  0.499 <BUG
dominance    eeg          rf      0.484  0.130  0.418 <BUG
dominance    peripheral   svm     0.524  0.114  0.508
dominance    peripheral   rf      0.521  0.088  0.451
dominance

In [4]:
# Class balance flags
any_flagged = False
for target in TARGETS:
    flagged = results['flagged_folds'][target]
    if flagged:
        print(f"[FLAG] {target}: imbalance >65/35 in subjects {flagged} — use F1 as primary metric for these folds")
        any_flagged = True
if not any_flagged:
    print("No flagged folds — all class ratios within 65/35.")

[FLAG] valence: imbalance >65/35 in subjects [3, 4, 6, 12, 17, 23, 25, 26, 27, 28, 29, 31] — use F1 as primary metric for these folds
[FLAG] arousal: imbalance >65/35 in subjects [2, 4, 8, 9, 15, 20, 23, 24, 26, 27, 28, 32] — use F1 as primary metric for these folds
[FLAG] dominance: imbalance >65/35 in subjects [4, 9, 16, 18, 20, 25, 27, 30, 32] — use F1 as primary metric for these folds


## 3. Importance aggregation

In [5]:
importance_agg = aggregate_importance(results['importance_mean'])

# Band-level (primary RQ1 result)
print("Band-level importance (summed across 32 electrodes, averaged across 32 folds):")
print(f"{'Band':<8}", '  '.join(f"{t:>10}" for t in TARGETS))
print('-' * 44)
for b, band in enumerate(BAND_NAMES):
    row = '  '.join(f"{importance_agg[t]['band'][b]:>10.5f}" for t in TARGETS)
    print(f"{band:<8}  {row}")

# Region-level
print("\nRegion-level importance:")
for target in TARGETS:
    print(f"\n  {target}:")
    for region, val in sorted(
        importance_agg[target]['region'].items(), key=lambda x: -x[1]
    ):
        print(f"    {region:<22} {val:.5f}")

Band-level importance (summed across 32 electrodes, averaged across 32 folds):
Band        valence     arousal   dominance
--------------------------------------------
theta       -0.02406    -0.02250     0.00434
alpha       -0.02910    -0.00773     0.01988
beta        -0.01469    -0.00141     0.04344
gamma       -0.03047    -0.00641    -0.00113

Region-level importance:

  valence:
    frontal_right          0.00605
    temporal_left          0.00168
    temporal_right         -0.00289
    occipital_right        -0.00422
    central                -0.01285
    occipital_left         -0.01504
    parietal_right         -0.01941
    parietal_left          -0.02164
    frontal_left           -0.03000

  arousal:
    occipital_left         0.00727
    temporal_left          0.00340
    temporal_right         0.00219
    central                -0.00340
    occipital_right        -0.00547
    parietal_left          -0.00750
    frontal_left           -0.01039
    frontal_right          -0.0

## 4. Importance correlation (RQ5)

In [6]:
imp_corr = importance_correlation(results['importance_mean'])

print("Pairwise Spearman importance correlation:")
print(f"{'Pair':<30} {'rho':>6}  {'p-value':>9}")
print('-' * 50)
for key, vals in imp_corr.items():
    print(f"{key:<30} {vals['rho']:>6.3f}  {vals['pvalue']:>9.4f}")

print("\n  Low rho for *_dominance pairs → PAD 3rd dimension supported")
print("  High rho for *_dominance pairs → Russell (1980) circumplex collapse supported")

Pairwise Spearman importance correlation:
Pair                              rho    p-value
--------------------------------------------------
valence_arousal                 0.124     0.1597
valence_dominance              -0.010     0.9070
arousal_dominance               0.006     0.9490

  Low rho for *_dominance pairs → PAD 3rd dimension supported
  High rho for *_dominance pairs → Russell (1980) circumplex collapse supported


## 5. Band ablation summary (RQ1 isolation)

In [7]:
print("Band ablation accuracy (RF, single band, LOSO mean):")
print(f"{'Band':<8}", '  '.join(f"{t:>10}" for t in TARGETS))
print('-' * 44)
for band in BAND_NAMES:
    row = '  '.join(
        f"{np.mean(results['ablation'][t][band]['acc']):>10.3f}"
        for t in TARGETS
    )
    print(f"{band:<8}  {row}")

print("\nFull model (EEG, RF):")
for target in TARGETS:
    print(f"  {target}: {np.mean(results['metrics'][target]['eeg']['rf']['acc']):.3f}")

Band ablation accuracy (RF, single band, LOSO mean):
Band        valence     arousal   dominance
--------------------------------------------
theta          0.498       0.489       0.498
alpha          0.512       0.488       0.482
beta           0.563       0.521       0.480
gamma          0.517       0.520       0.495

Full model (EEG, RF):
  valence: 0.527
  arousal: 0.537
  dominance: 0.484


## 6. Save all LOSO outputs

In [8]:
with open('../results/loso_results.pkl', 'wb') as f:
    pickle.dump(results, f)

with open('../results/importance_agg.pkl', 'wb') as f:
    pickle.dump(importance_agg, f)

with open('../results/importance_corr.pkl', 'wb') as f:
    pickle.dump(imp_corr, f)

for target in TARGETS:
    np.save(f'../results/importance_mean_{target}.npy',
            results['importance_mean'][target])

print("Saved: loso_results.pkl, importance_agg.pkl, importance_corr.pkl")
print("Saved: importance_mean_{valence,arousal,dominance}.npy")

sizes = {p.name: f"{p.stat().st_size/1e6:.1f} MB"
         for p in Path('../results').glob('*.pkl')}
for name, size in sizes.items():
    print(f"  {name}: {size}")

Saved: loso_results.pkl, importance_agg.pkl, importance_corr.pkl
Saved: importance_mean_{valence,arousal,dominance}.npy
  loso_results.pkl: 0.1 MB
  importance_corr.pkl: 0.0 MB
  importance_agg.pkl: 0.0 MB


## 7. Null distribution

Batched with checkpoint saving to `results/null_checkpoints/`.
Safe to interrupt — resumes from last completed batch on rerun.

Settings: 100 permutations, batch_size=10, 100 trees, 10 perm repeats.
These are already the reduced settings for the null — 100 permutations of
32-fold averaging compensates for the looser per-permutation estimates.

Expected runtime: 30-50 min total across 10 batches.

In [14]:
assert Path('../results/loso_results.pkl').exists(), \
    "Run Section 6 first — LOSO results must be saved before starting null."

null_results = shuffled_label_null(
    X_eeg            = X_features,
    X_peripheral     = X_peripheral_features,
    y                = y,
    subject_ids      = subject_ids,
    n_permutations   = 100,
    batch_size       = 10,
    checkpoint_dir   = '../results/null_checkpoints',
    rf_n_estimators  = 100,
    n_perm_repeats   = 10,
    n_jobs           = -3,
)
print("Null distribution complete.")

Null batches:   0%|          | 0/10 [00:00<?, ?batch/s]

Batch 1/10: resumed from checkpoint.
Batch 2/10: resumed from checkpoint.
Batch 3/10: resumed from checkpoint.
Batch 4/10: resumed from checkpoint.
Batch 5/10: resumed from checkpoint.
Batch 6/10: resumed from checkpoint.
Batch 7/10: resumed from checkpoint.
Batch 8/10: resumed from checkpoint.
Batch 9/10: resumed from checkpoint.
Batch 10/10: running 10 permutations...


  Batch 10 perms:   0%|          | 0/10 [00:00<?, ?perm/s]

  Checkpoint saved: batch_009.pkl
Null distribution complete.


In [15]:
# Observed vs null comparison
print("Null accuracy (EEG, RF) — mean | std | p95:")
for target in TARGETS:
    s   = null_results['accuracy'][target]['eeg']
    obs = np.mean(results['metrics'][target]['eeg']['rf']['acc'])
    sig = 'ABOVE NULL' if obs > s['p95'] else 'within null'
    print(f"  {target:<12} null={s['mean']:.3f}±{s['std']:.3f} p95={s['p95']:.3f} "
          f"obs={obs:.3f}  → {sig}")

print("\nImportance correlation vs null:")
for key in CORR_KEYS:
    s   = null_results['correlation'][key]
    obs = imp_corr[key]['rho']
    sig = 'ABOVE NULL' if obs > s['p95'] else 'within null'
    print(f"  {key:<30} null_p95={s['p95']:.3f}  obs={obs:.3f}  → {sig}")

Null accuracy (EEG, RF) — mean | std | p95:
  valence      null=0.501±0.027 p95=0.541 obs=0.527  → within null
  arousal      null=0.508±0.022 p95=0.543 obs=0.537  → within null
  dominance    null=0.498±0.018 p95=0.530 obs=0.484  → within null

Importance correlation vs null:
  valence_arousal                null_p95=0.140  obs=0.124  → within null
  valence_dominance              null_p95=0.190  obs=-0.010  → within null
  arousal_dominance              null_p95=0.146  obs=0.006  → within null


In [16]:
with open('../results/null_results.pkl', 'wb') as f:
    pickle.dump(null_results, f)

print("Saved: null_results.pkl")
print("All results saved. Ready for 04_results.ipynb.")

Saved: null_results.pkl
All results saved. Ready for 04_results.ipynb.
